In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from generator import RoadNetwork, Trajectory
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd
import networkx as nx
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch_geometric.transforms as T

from models import SpatialFlowConvolution

In [ ]:
city = "porto"
city_traj = "porto"
device_nr = "3"

In [ ]:
test = pd.read_pickle(
        f"../../datasets/trajectories/{city}/traj_train_test_split/test_69.pkl"
    )
test["seg_seq"] = test["seg_seq"].map(np.array)

In [ ]:
network = RoadNetwork()
network.load(f"../../osm_data/{city}")
trajectory = Trajectory(f"../../datasets/trajectories/{city_traj}/road_segment_map_final.csv", nrows=100000000).generate_TTE_datatset()

traj_features = pd.read_csv(f"../../datasets/trajectories/{city_traj}/speed_features_unnormalized.csv")
traj_features.set_index(["u", "v", "key"], inplace=True)
traj_features["util"] = (traj_features["util"] - traj_features["util"].min()) / (traj_features["util"].max() - traj_features["util"].min())  # min max normalization
traj_features["avg_speed"] = (traj_features["avg_speed"] - traj_features["avg_speed"].min()) / (traj_features["avg_speed"].max() - traj_features["avg_speed"].min())  # min max normalization
traj_features.fillna(0, inplace=True)

# data = network.generate_road_segment_pyg_dataset(drop_labels=["highway_enc"])

In [ ]:
data = network.generate_road_segment_pyg_dataset(include_coords=True, dataset=city)

In [ ]:
#adj = np.loadtxt(f"./sfc_precalc_adj/traj_adj_k_2_{city}.gz") # for traj2vec 'traj_adj_k_1_False_no_selfloops_smoothed'

In [ ]:
device = torch.device(f'cuda:{device_nr}' if torch.cuda.is_available() else 'cpu')
# precalc adj matrices
SpatialFlowConvolution(data, device, network, trajectory, k=2, bidirectional=False, add_self_loops=True)

In [ ]:

adj = np.loadtxt(f"./sfc_precalc_adj/traj_adj_k_2_False_{city}.gz")

In [ ]:
models = []
device = torch.device(f'cuda:{device_nr}' if torch.cuda.is_available() else 'cpu')
model = SpatialFlowConvolution(data, device, network, adj=adj)

In [ ]:
model.train(epochs=10000)

In [ ]:
z = model.load_emb()
z.shape

In [ ]:
model.save_model(path="../model_states/sfc/")